In [1]:
!nvidia-smi
!rm -rf /kaggle/working/repo
!git clone https://github.com/rubinanoor/small-model-alignment-starter.git /kaggle/working/repo
%cd /kaggle/working/repo
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest peft trl
!python -m pip install -q -e . --no-deps
import torch, transformers, datasets
print(torch.__version__, transformers.__version__, datasets.__version__, torch.cuda.device_count())

Thu Oct 15 20:54:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!cd /kaggle/working/repo && PYTHONPATH=. python -m pytest -q tests

............                                                             [100%]
12 passed in 6.65s


In [3]:
import os, shutil, glob
hits = [h for h in glob.glob("/kaggle/input/**/sft/best", recursive=True) if os.path.exists(f"{h}/model.safetensors")]
print(hits)
SRC_BEST = hits[0]
os.makedirs("/kaggle/working/repo/outputs/sft", exist_ok=True)
shutil.copytree(SRC_BEST, "/kaggle/working/repo/outputs/sft/best", dirs_exist_ok=True)
!ls /kaggle/working/repo/outputs/sft/best

['/kaggle/input/notebooks/rubinanoor/assignment-2-notebook-1/repo/outputs/sft/best']
added_tokens.json	merges.txt		 tokenizer.json
chat_template.jinja	model.safetensors	 vocab.json
config.json		special_tokens_map.json
generation_config.json	tokenizer_config.json


In [4]:
!cd /kaggle/working/repo && PYTHONPATH=. python scripts/train_dpo.py --dataset uf --out outputs/dpo_smoke --n_train 128 --n_val 16 --accum 4 --eval_every 2

The tokenizer you are loading from 'outputs/sft/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
README.md: 6.53kB [00:00, 19.2MB/s]
data/train_prefs-00000-of-00001.parquet: 100%|█| 226M/226M [00:01<00:00, 126MB/s
data/test_prefs-00000-of-00001.parquet: 100%|█| 7.29M/7.29M [00:01<00:00, 6.38MB
data/test_sft-00000-of-00001.parquet: 100%|█| 3.72M/3.72M [00:00<00:00, 16.5MB/s
data/train_gen-00000-of-00001.parquet: 100%|██| 184M/184M [00:01<00:00, 137MB/s]
data/test_gen-00000-of-00001.parquet: 100%|█| 3.02M/3.02M [00:00<00:00, 3.92MB/s
Generating train_prefs split: 100%|█| 61135/61135 [00:01<00:00, 58969.65 example
Generating train_sft split: 100%|█| 61135/61135 [00:00<00:00, 70973.53 examples/
Generating t

In [5]:
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv
!rm -rf /kaggle/working/repo/outputs/dpo_smoke

index, memory.used [MiB], memory.total [MiB]
0, 0 MiB, 15360 MiB
1, 0 MiB, 15360 MiB


In [6]:
# start clean: metrics.jsonl is appended to, so a leftover file from a failed attempt would mix two runs
!rm -rf /kaggle/working/repo/outputs/dpo_uf
!cd /kaggle/working/repo && PYTHONPATH=. python scripts/train_dpo.py --dataset uf --out outputs/dpo_uf

The tokenizer you are loading from 'outputs/sft/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
train pairs=15000 val pairs=1000 steps/epoch=468
step 1/468 loss=0.6931 acc=0.000 margin=0.0000 gn=69.46 | VAL loss=0.6897 acc=0.566 margin=0.0073 [2.9 min, peak 9.2 GB]
step 2/468 loss=0.6879 acc=0.406 margin=0.0111 gn=68.28 [3.2 min, peak 9.2 GB]
step 3/468 loss=0.6981 acc=0.406 margin=-0.0081 gn=66.05 [3.5 min, peak 9.2 GB]
step 4/468 loss=0.6881 acc=0.531 margin=0.0119 gn=57.39 [3.8 min, peak 9.2 GB]
step 5/468 loss=0.6700 acc=0.688 margin=0.0495 gn=66.08 [4.1 min, peak 9.2 GB]
step 6/468 loss=0.6911 acc=0.562 margin=0.0058 gn=62.05 [4.4 min, peak 9.2 GB]
step 7/468 loss=0.6866 acc=0.625 margin=0.0178 gn

In [7]:
!grep val_loss /kaggle/working/repo/outputs/dpo_uf/metrics.jsonl
!ls /kaggle/working/repo/outputs/dpo_uf /kaggle/working/repo/outputs/dpo_uf/best

{"step": 0, "train_loss": 0.6931471824645996, "train_acc": 0.0, "train_margin": 0.0, "grad_norm": 69.45635223388672, "val_loss": 0.6897406457662583, "val_acc": 0.566, "val_margin": 0.007256528796162456, "saved_best": true}
{"step": 100, "train_loss": 0.6365037593059242, "train_acc": 0.65625, "train_margin": 0.2039660442096647, "grad_norm": 57.25239181518555, "val_loss": 0.6056906319260598, "val_acc": 0.68, "val_margin": 0.3138256509564817, "saved_best": true}
{"step": 200, "train_loss": 0.5409771013073623, "train_acc": 0.75, "train_margin": 0.4671520603587851, "grad_norm": 48.971866607666016, "val_loss": 0.5904667264521122, "val_acc": 0.698, "val_margin": 0.4061210328191519, "saved_best": true}
{"step": 300, "train_loss": 0.5356296704849228, "train_acc": 0.6875, "train_margin": 0.5304987728595734, "grad_norm": 53.694786071777344, "val_loss": 0.5833183993026614, "val_acc": 0.711, "val_margin": 0.4466263347864151, "saved_best": true}
{"step": 400, "train_loss": 0.5911980763776228, "train

In [8]:
!rm -rf /kaggle/working/repo/outputs/sft
!du -sh /kaggle/working/repo/outputs

3.8G	/kaggle/working/repo/outputs
